# ĐỒ ÁN PHÂN TÍCH DỮ LIỆU
## QUY TRÌNH 2.1.1: TIỀN XỬ LÝ DỮ LIỆU - XỬ LÝ BẢN GHI TRÙNG LẶP & CHUẨN HÓA ĐỊNH DẠNG
- **Thành viên thực hiện:** Trần Nhựt Hào
- **Nhiệm vụ theo phân công:**
  1. Xử lý bản ghi trùng lặp (Duplicate Records Removal).
  2. Chuẩn hóa định dạng: Cột Date (về datetime YYYY-MM-DD), Cột Amount (chuẩn hóa các chuỗi $, ngoặc âm, kiểu số VN/Quốc tế thành float), Chuẩn hóa nhãn các biến phân loại Channel, Region, Revenue or expense.

In [ ]:
# Kết nối Google Drive nếu đang chạy trên Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass  # Chạy trên môi trường cục bộ (Jupyter Notebook / VS Code)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, spearmanr

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Đường dẫn đến thư mục dữ liệu thô (tự động nhận diện local hoặc Google Colab)
RAW = Path("RAW_DATA/raw") if Path("RAW_DATA/raw").exists() else Path("/content/drive/MyDrive/[LTPTDL] Đồ án/raw")

FILES = [
    "Fact", "Calendar", "Business Line", "Group", "Subgroup",
    "Channel", "Region", "Headlines", "KPI", "Ratios"
]

# Đọc toàn bộ bảng dữ liệu dưới dạng chuỗi
tables = {
    f: pd.read_csv(RAW / f"{f}.csv", dtype=str, encoding="utf-8-sig")
    for f in FILES
}

fact = tables["Fact"]

# Kiểm tra kích thước các bảng
{name: df.shape for name, df in tables.items()}

# 2. Data Preprocessing

## 2.1. Data cleaning

### Bước 1 & Bước 2: Xử lý bản ghi trùng lặp và Chuẩn hóa định dạng

In [ ]:
# Định nghĩa hàm chuẩn hóa số Amount
def parse_amount(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    neg = s.startswith("(") or s.startswith("-")
    s = re.sub(r"USD|\$|[()\s-]", "", s, flags=re.I)
    if re.fullmatch(r"\d{1,3}(?:\.\d{3})+,\d{2}|\d+,\d{2}", s):   # kiểu VN: 6.720,07
        s = s.replace(".", "").replace(",", ".")
    else:                                                          # kiểu quốc tế: 1,632.79
        s = s.replace(",", "")
    return (-1 if neg else 1) * float(s)

# Tạo bản sao, giữ nguyên dữ liệu gốc
df = fact.copy()

# Đổi tên cột cho dễ thao tác
df = df.rename(columns={
    "Amount, $": "Amount",
    "Business Line ID": "BL_ID",
    "Business Line Name": "BL_Name",
    "Subgroup ID": "Subgroup_ID",
    "Group ID": "Group_ID",
    "Revenue or expense": "RevExp"
})

# Bảng chuẩn dùng để sửa khóa
SG = tables["Subgroup"]
BL = tables["Business Line"].copy()
n_goc = len(df)

# 1. Xóa dòng trùng (Thành viên: Trần Nhựt Hào)
df = df.drop_duplicates().reset_index(drop=True)

# 2. Chuẩn hoá định dạng (Thành viên: Trần Nhựt Hào)
# 2.1 Date:
df["Date"] = pd.to_datetime(
    df["Date"], format="%Y-%m-%d", errors="coerce"
).fillna(
    pd.to_datetime(df["Date"], format="%d/%m/%Y", errors="coerce")
).fillna(
    pd.to_datetime(df["Date"], format="%Y%m%d", errors="coerce")
).fillna(
    pd.to_datetime(df["Date"], format="%b %d, %Y", errors="coerce")
)

# 2.2 Amount: đưa về số thực
df["Amount"] = df["Amount"].map(parse_amount)

# 2.3 Chuẩn hoá nhãn phân loại
df["Channel"] = df["Channel"].str.strip().str.lower().map({
    "online": "Online", "store": "Store", "distributor": "Distributor"
})

df["Region"] = (
    df["Region"].str.strip().str.lower()
    .str.replace("miền ", "", regex=False)
    .map({"bắc": "Bắc", "bac": "Bắc", "trung": "Trung", "nam": "Nam"})
)

df["RevExp"] = (
    df["RevExp"].str.strip().str.lower()
    .map({
        "revenue": "Revenue", "rev": "Revenue",
        "expense": "Expense", "expenses": "Expense", "exp": "Expense"
    })
)

In [ ]:
# Kiểm tra kết quả sau khi Xử lý trùng lặp và Chuẩn hóa định dạng
print(f"Số dòng ban đầu: {n_goc} dòng")
print(f"Số dòng sau xóa trùng: {len(df)} dòng (Đã giảm: {n_goc - len(df)} dòng trùng lặp)")
print("Số dòng trùng lặp hoàn toàn còn lại:", df.duplicated().sum())
print("\nThông tin dữ liệu sau chuẩn hóa định dạng:")
print(df.info())
display(df.head(10))

**Nhận xét:**
- Bảng Fact sau khi xóa các dòng trùng lặp đã loại bỏ đúng 86 bản ghi trùng lặp hoàn toàn (tương ứng 0,99%).
- Các định dạng ngày tháng không đồng nhất (YYYY-MM-DD, DD/MM/YYYY, YYYYMMDD, MMM D, YYYY) đã được chuyển đổi thành công về chuẩn `datetime64[ns]`.
- Cột Amount đã được chuyển đổi từ các định dạng chuỗi phức tạp sang kiểu số thực `float64`.
- Nhãn của các biến phân loại `Channel`, `Region`, `RevExp` đã được loại bỏ khoảng trắng thừa, đưa về chữ thường và ánh xạ chính xác về các nhãn chuẩn đồng nhất.